In [2]:
from cobra.io	import read_sbml_model
from cobra.medium	import minimal_medium

/Users/edwin/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
ecoli_model = read_sbml_model('/Users/edwin/bigg_models/iML1515.xml')

In [4]:
ecoli_model.medium

{'EX_pi_e': 1000.0,
 'EX_co2_e': 1000.0,
 'EX_fe3_e': 1000.0,
 'EX_h_e': 1000.0,
 'EX_mn2_e': 1000.0,
 'EX_fe2_e': 1000.0,
 'EX_glc__D_e': 10.0,
 'EX_zn2_e': 1000.0,
 'EX_mg2_e': 1000.0,
 'EX_ca2_e': 1000.0,
 'EX_ni2_e': 1000.0,
 'EX_cu2_e': 1000.0,
 'EX_sel_e': 1000.0,
 'EX_cobalt2_e': 1000.0,
 'EX_h2o_e': 1000.0,
 'EX_mobd_e': 1000.0,
 'EX_so4_e': 1000.0,
 'EX_nh4_e': 1000.0,
 'EX_k_e': 1000.0,
 'EX_na1_e': 1000.0,
 'EX_cl_e': 1000.0,
 'EX_o2_e': 1000.0,
 'EX_tungs_e': 1000.0,
 'EX_slnt_e': 1000.0}

In [5]:
# Get the maximum possible growth rate (or set your desired target rate)
max_growth = ecoli_model.slim_optimize()
print(f'Maximum growth rate: {max_growth}')

# Calculate the medium with the lowest total import flux
min_medium = minimal_medium(ecoli_model, max_growth)
print(min_medium)

Maximum growth rate: 0.87699721425716
EX_pi_e          0.845957
EX_mn2_e         0.000606
EX_fe2_e         0.014085
EX_glc__D_e     10.000000
EX_zn2_e         0.000299
EX_mg2_e         0.007608
EX_ca2_e         0.004565
EX_ni2_e         0.000283
EX_cu2_e         0.000622
EX_cobalt2_e     0.000022
EX_mobd_e        0.000006
EX_so4_e         0.220845
EX_nh4_e         9.471495
EX_k_e           0.171184
EX_cl_e          0.004565
EX_o2_e         22.131763
dtype: float64


In [120]:
lb_medium = {
				"EX_pi_e": 1000.0,   # effectively unconstrained phosphate uptake
    "EX_ni2_e": 10000.0, 	# nickel, often required by biomass
    "EX_so4_e": 10000.0,  # sulfate, often required by biomass
    "EX_o2_e":23.0,       #	oxygen, often required by biomass
			 "EX_na1_e": 1000.0,    # sodium from NaCl
    "EX_cl_e": 1000.0,     # chloride from NaCl
    "EX_k_e": 1000.0,      # potassium, often required by biomass
    "EX_mg2_e": 1000.0,    # magnesium
    "EX_ca2_e": 1000.0,    # calcium
    "EX_fe2_e": 1000.0,    # iron
    "EX_mn2_e": 1000.0,    # manganese
    "EX_zn2_e": 1000.0,    # zinc
    "EX_cu2_e": 1000.0,    # copper
    "EX_cobalt2_e": 1000.0, # cobalt
    "EX_mobd_e": 1000.0,   # molybdate

    # Amino-acid-rich tryptone/casein hydrolysate approximation
    # "EX_ala__L_e": 1000.0,
    # "EX_arg__L_e": 1000.0,
    # "EX_asn__L_e": 1000.0,
    # "EX_asp__L_e": 1000.0,
    # "EX_cys__L_e": 1000.0,
    # "EX_gln__L_e": 1000.0,
    "EX_glu__L_e": 50.0,
    #"EX_gly_e": 1000.0,
    # "EX_his__L_e": 1000.0,
    # "EX_ile__L_e": 1000.0,
    # "EX_leu__L_e": 1000.0,
    # "EX_lys__L_e": 1000.0,
    # "EX_met__L_e": 1000.0,
    # "EX_phe__L_e": 1000.0,
    # "EX_pro__L_e": 1000.0,
    "EX_ser__L_e": 50.0,
    #"EX_thr__L_e": 50.0,
    "EX_trp__L_e": 50.0,
    # "EX_tyr__L_e": 1000.0,
    # "EX_val__L_e": 1000.0,

    # Yeast-extract-like vitamins/growth factors
    "EX_thm_e": 1000.0,
    "EX_nac_e": 1000.0,
    "EX_pnto__R_e": 1000.0,
    "EX_pydx_e": 1000.0,
    "EX_btn_e": 1000.0,
} 

In [121]:
len(lb_medium)

23

In [122]:
ecoli_model.medium = lb_medium

In [123]:
sol = ecoli_model.optimize()
print(sol.objective_value)

1.2994390959368736


In [137]:
for rxn in ecoli_model.reactions:
    rxn_name = rxn.name

    if "manganese" in rxn_name.lower():
        print(sol.fluxes[rxn.id], ":", rxn.id,rxn_name)

0.0 : MNt2pp Manganese (Mn+2) transport in via proton symport (periplasm)
0.0008979124152923798 : MNtex Manganese (Mn+2) transport via diffusion (extracellular to periplasm)
0.0008979124152923798 : MN2tpp Manganese transport in via permease (no H+)
0.0 : MN2t3pp Manganese (Mn+2) transport out via proton antiport (periplasm)
0.0 : MN2tipp Manganese transport in via permease (no H+)


In [126]:
sol.fluxes["EX_na1_e"]

5.079568382631115e-15

In [125]:
# Get the maximum possible growth rate (or set your desired target rate)
max_growth = ecoli_model.slim_optimize()
print(f'Maximum growth rate: {max_growth}')

# Calculate the medium with the lowest total import flux
min_medium = minimal_medium(ecoli_model, max_growth)
print(min_medium)

Maximum growth rate: 1.2994390959368736
EX_pi_e          1.253447
EX_mn2_e         0.000898
EX_glu__L_e     29.779729
EX_btn_e         0.000003
EX_fe2_e         0.020870
EX_ser__L_e     50.000000
EX_thm_e         0.000290
EX_trp__L_e      0.073864
EX_zn2_e         0.000443
EX_mg2_e         0.011273
EX_nac_e         0.002960
EX_ca2_e         0.006764
EX_ni2_e         0.000420
EX_cu2_e         0.000921
EX_cobalt2_e     0.000032
EX_pnto__R_e     0.000876
EX_mobd_e        0.000009
EX_so4_e         0.326930
EX_k_e           0.253641
EX_cl_e          0.006764
EX_pydx_e        0.000290
EX_o2_e         23.000000
dtype: float64


In [84]:
len(min_medium)

22

In [ ]:
for rxn_id in min_medium.keys():
    rxn_name = ecoli_model.reactions.get_by_id(rxn_id).name
    print(rxn_id, ":", rxn_name)

EX_pi_e : Phosphate exchange
EX_mn2_e : Mn2+ exchange
EX_glu__L_e : L-Glutamate exchange
EX_btn_e : Biotin exchange
EX_fe2_e : Fe2+ exchange
EX_ser__L_e : L-Serine exchange
EX_thm_e : Thiamin exchange
EX_trp__L_e : L-Tryptophan exchange
EX_zn2_e : Zinc exchange
EX_mg2_e : Mg exchange
EX_nac_e : Nicotinate exchange
EX_ca2_e : Calcium exchange
EX_ni2_e : Ni2+ exchange
EX_cu2_e : Cu2+ exchange
EX_cobalt2_e : Co2+ exchange
EX_pnto__R_e : (R)-Pantothenate exchange
EX_mobd_e : Molybdate exchange
EX_so4_e : Sulfate exchange
EX_k_e : K+ exchange
EX_cl_e : Chloride exchange
EX_pydx_e : Pyridoxal exchange
EX_o2_e : O2 exchange
